# FlashEats — Class 5 Investigation

## Client escalation

> **“Late deliveries are increasing and customers say our ETA is unreliable. Figure out what is happening before we invest in an AI delay-prediction system.”**

Today is not a syntax lesson. Your job is to use SQL, CSV, JSON and APIs like an FDE:

**define the question → retrieve evidence → validate completeness → reconcile sources → state what you know and what you still cannot know**

### Rules
- Do not train an ML model.
- State your definition of **late** before calculating it.
- Preserve raw API responses.
- Do not silently drop failures.
- If another team gets a different answer, investigate the definition and data grain first.

In [ ]:
# 0. INSTALLS + IMPORTS
!pip -q install pandas requests flask matplotlib

import os, sys, json, time, sqlite3, subprocess, zipfile
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Environment ready.")

In [ ]:
# Local discovery: check if database exists in current working directory
BASE = None
if (Path.cwd() / 'database' / 'flasheats.db').exists():
    BASE = Path.cwd()
elif (Path.cwd().parent / 'database' / 'flasheats.db').exists():
    BASE = Path.cwd().parent

if BASE is None:
    BASE = Path('/content/FlashEats_Classroom_Pack')

# 1. LOAD THE CLASSROOM PACK (Colab fallback)
if not BASE.exists():
    try:
        from google.colab import files
        print('Upload FlashEats_Classroom_Pack.zip')
        uploaded = files.upload()
        zip_name = next(name for name in uploaded if name.endswith('.zip'))
        with zipfile.ZipFile(zip_name, 'r') as z:
            z.extractall('/content')
    except Exception as e:
        print('If running locally, set BASE manually to the extracted pack.')
        print(e)

print('BASE:', BASE)
print('Exists:', BASE.exists())


## 2. Source map first

Before analysis, predict where you would look for:

| Information | Likely source | Why? |
|---|---|---|
| Promised delivery time | `database/flasheats.db` (`orders`) | Operational system of record presented to customer at checkout |
| Actual delivery time | `database/flasheats.db` (`orders`) | Final drop-off timestamp recorded upon completion |
| Driver assignment | Mock Dispatch API / `driver_events.json` | Real-time dispatching engine and driver tracking system |
| Customer complaint | `data/support_tickets.csv` | Customer contact log with category and ticket description |
| Restaurant status | `data/restaurant_status.csv` | Kitchen display system status updates (preparing, ready, handed_off) |
| Driver movement/events | `data/driver_events.json` | Driver mobile app GPS pings and state transitions |

**Discuss:** Which source is authoritative for each fact, and which is only contextual?
- **Authoritative**: `orders` for contractual order times (created, promised, actual); Dispatch API for official driver assignment.
- **Contextual**: `support_tickets.csv` captures customer perception (not measured duration); `restaurant_status.csv` reflects manual kitchen updates that may lag physical events.


In [ ]:
# 3. BASIC SYNTAX — READ EVERY SOURCE

# SQLite
db_path = BASE / "database" / "flasheats.db"
con = sqlite3.connect(db_path)

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    con
)
display(tables)

sample_orders = pd.read_sql("SELECT * FROM orders LIMIT 5;", con)
display(sample_orders)

# CSV
tickets = pd.read_csv(BASE / "data" / "support_tickets.csv")
restaurant_status = pd.read_csv(BASE / "data" / "restaurant_status.csv")
display(tickets.head())

# Nested JSON
with open(BASE / "data" / "driver_events.json", "r") as f:
    driver_events = json.load(f)

print("Driver objects:", len(driver_events))
display(driver_events[:1])

# Challenge 1 — How large is the late-delivery problem?

Before coding, decide:

- **Late means:** An order where `actual_delivery_at > promised_eta`.
- **Denominator:** Delivered orders with valid, non-null `actual_delivery_at` (1,495 orders).
- **Cancelled orders:** Excluded from delivery delay metrics (68 orders), analyzed separately as lost orders.
- **Missing actual delivery timestamp:** Excluded from delay calculations (37 delivered orders), flagged as a data-logging defect.
- **Row grain:** One unique customer order (`order_id`).

### Required analysis
1. valid delivered orders,
2. late count and percentage,
3. median lateness among late orders,
4. worst delayed orders,
5. any data-quality issue that changes the metric.


In [ ]:
# Challenge 1: Late delivery analysis
query = '''
SELECT 
    order_id,
    customer_id,
    restaurant_id,
    driver_id,
    created_at,
    promised_eta,
    pickup_at,
    actual_delivery_at,
    final_status,
    distance_km_estimate,
    traffic_bucket,
    weather_bucket
FROM orders;
'''
orders = pd.read_sql(query, con)

# 1. Deduplicate by order_id (there are 3 duplicate rows)
orders = orders.drop_duplicates(subset=['order_id'], keep='first')

# 2. Parse datetime columns
for col in ['created_at', 'promised_eta', 'pickup_at', 'actual_delivery_at']:
    orders[col] = pd.to_datetime(orders[col], format='mixed', errors='coerce')

# 3. Compute delay in minutes
orders['delay_min'] = (orders['actual_delivery_at'] - orders['promised_eta']).dt.total_seconds() / 60

# 4. Filter delivered orders with recorded actual delivery time
delivered = orders[(orders['final_status'].str.lower() == 'delivered') & (orders['actual_delivery_at'].notna())].copy()
late_orders = delivered[delivered['delay_min'] > 0]

print('=== CHALLENGE 1 METRICS ===')
print(f'Total unique orders: {len(orders)}')
print(f'Cancelled orders: {(orders["final_status"].str.lower() == "cancelled").sum()}')
print(f'Delivered orders missing completion timestamp: {orders[orders["final_status"].str.lower() == "delivered"]["actual_delivery_at"].isna().sum()}')
print(f'Valid delivered orders (denominator): {len(delivered)}')
print(f'Late delivered orders: {len(late_orders)}')
print(f'Late Delivery Rate: {len(late_orders) / len(delivered):.1%}')
print(f'Median delay among late orders: {late_orders["delay_min"].median():.1f} min')
print(f'Mean delay among late orders: {late_orders["delay_min"].mean():.1f} min')

print('\nTop 5 worst delayed orders:')
display(late_orders[['order_id', 'promised_eta', 'actual_delivery_at', 'delay_min']].sort_values('delay_min', ascending=False).head(5))


## Checkpoint

Compare your answer with another team.

If your counts differ, investigate:
- duplicate rows,
- cancelled orders,
- missing timestamps,
- late-delivery definition,
- row grain.

# Challenge 2 — Operations says: “Traffic is the problem.”

Test that claim using at least three dimensions.

### Required output
- 2–3 comparisons (traffic, weather, kitchen prep time),
- one hypothesis supported by evidence,
- one alternative explanation you cannot rule out.

| Dimension | Median Delay (min) | Mean Delay (min) | Correlation with Delay |
|---|---:|---:|---:|
| Traffic (Severe vs Low) | 4.3 vs -0.1 | 6.3 vs 1.2 | 0.21 (transit time) |
| Weather (Heavy Rain vs Clear) | 6.6 vs 0.8 | 7.7 vs 2.2 | - |
| Kitchen Prep Time (`pickup - created`) | 26.1 | 28.5 | **0.83** |
| Transit Time (`delivery - pickup`) | 47.2 | 48.1 | 0.21 |

- **Hypothesis supported by evidence:** Kitchen preparation delay is the primary driver of late deliveries (0.83 correlation with delay). Traffic and weather add marginal friction, but orders are already late before the driver leaves the restaurant.
- **Alternative explanation we cannot rule out:** Drivers may be arriving early at restaurants and waiting for food, or the restaurant may mark food ready before it is actually handed off. We cannot separate driver transit to restaurant from kitchen cooking time because there is no recorded restaurant-arrival timestamp.


In [ ]:
# Challenge 2: Test Operations traffic claim
delivered['prep_min'] = (delivered['pickup_at'] - delivered['created_at']).dt.total_seconds() / 60
delivered['transit_min'] = (delivered['actual_delivery_at'] - delivered['pickup_at']).dt.total_seconds() / 60

print('1. Delay by Traffic Bucket:')
display(delivered.groupby('traffic_bucket')['delay_min'].agg(['count', 'median', 'mean']).round(1))

print('\n2. Delay by Weather Bucket:')
display(delivered.groupby('weather_bucket')['delay_min'].agg(['count', 'median', 'mean']).round(1))

print('\n3. Correlation Matrix with Delay:')
display(delivered[['delay_min', 'prep_min', 'transit_min', 'distance_km_estimate']].corr().round(2))


# Challenge 3 — Customer Support disagrees

Use `support_tickets.csv`.

### Questions and Answers
1. **What are customers actually complaining about?** Orders arriving past promised ETA, ETAs constantly shifting later in the app, food arriving cold because it sat ready at the restaurant, and drivers not moving.
2. **Which complaint types are most common?** `late_delivery` (38 tickets), `eta_changed` (37 tickets), `restaurant_delay` (37 tickets), and `ready_but_waiting` (33 tickets).
3. **Are ticket IDs unique?** No, there is 1 duplicate `ticket_id` (202 total rows, 201 unique ticket IDs).
4. **Does the customer story match the operational story?** No. Operations claims external traffic is to blame, but customers specifically report changing ETAs, restaurant preparation stalls, and food sitting ready while waiting for drivers.
5. **What can tickets tell you that timestamps cannot?** Tickets reveal customer frustration, perceived unreliability (moving ETAs), and physical quality issues (cold food), none of which are captured by simple delivery duration numbers.

### System View vs Customer View
- **System View:** 'Order O00120 was delivered with a minor 6-minute delay.'
- **Customer View:** 'The ETA changed three times, the app said the food was ready 20 minutes ago, and my meal arrived cold.'


In [ ]:
# Challenge 3: Customer Support ticket analysis
print('Total tickets:', len(tickets))
print('Unique ticket IDs:', tickets['ticket_id'].nunique())
print('Duplicate ticket IDs count:', tickets['ticket_id'].duplicated().sum())

print('\nCleaned complaint categories:')
clean_categories = tickets['category'].str.strip().str.lower()
display(clean_categories.value_counts())

# Merge tickets with order delays
ticket_delay = tickets.merge(delivered[['order_id', 'delay_min', 'traffic_bucket']], on='order_id', how='left')
print('\nMedian delay by ticket complaint category:')
display(ticket_delay.groupby('category')['delay_min'].agg(['count', 'median']).round(1))


# Challenge 4 — Dispatch API: prove you retrieved everything

Your task:
1. start the API,
2. inspect one response,
3. retrieve **all** records,
4. handle pagination,
5. handle retryable failures,
6. save every successful raw page,
7. prove ingestion completeness.

A successful HTTP `200` on one page does **not** prove the job succeeded.

In [ ]:
# Start the mock API
api_script = BASE / "api" / "mock_dispatch_api.py"

api_proc = subprocess.Popen(
    [sys.executable, str(api_script)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)
health = requests.get("http://127.0.0.1:8000/health", timeout=10)
print(health.status_code, health.json())

In [ ]:
# Basic API syntax
API_URL = "http://127.0.0.1:8000/dispatch/orders"

response = requests.get(
    API_URL,
    params={"page": 1, "page_size": 50},
    timeout=10
)

print("HTTP:", response.status_code)

if response.ok:
    payload = response.json()
    print("Keys:", payload.keys())
    print("Records on page:", len(payload["data"]))
    print("Has more:", payload["has_more"])
    print("Reported total:", payload["total_records"])

In [ ]:
# Challenge 4: Reliable paginated ingestion with retry and raw payload persistence
RAW_DIR = BASE / 'student_output' / 'raw_dispatch'
RAW_DIR.mkdir(parents=True, exist_ok=True)

def fetch_all_dispatch_orders():
    records = []
    page = 1
    page_size = 50
    
    while True:
        url = 'http://127.0.0.1:8000/dispatch/orders'
        # Retry transient HTTP 500 and rate limit HTTP 429
        for attempt in range(5):
            try:
                resp = requests.get(url, params={'page': page, 'page_size': page_size}, timeout=10)
                if resp.status_code in [429, 500]:
                    time.sleep(1)
                    continue
                resp.raise_for_status()
                break
            except requests.RequestException as e:
                if attempt == 4:
                    raise RuntimeError(f'Failed on page {page}: {e}')
                time.sleep(1)
                
        payload = resp.json()
        
        # Save raw page to disk
        raw_path = RAW_DIR / f'page_{page:03d}.json'
        raw_path.write_text(json.dumps(payload, indent=2))
        
        records.extend(payload.get('data', []))
        
        if not payload.get('has_more', False):
            break
        page += 1
        
    return records

dispatch_records = fetch_all_dispatch_orders()
print(f'Retrieved total dispatch records: {len(dispatch_records)}')
print(f'Preserved {len(list(RAW_DIR.glob("page_*.json")))} raw JSON pages in {RAW_DIR}')


# Challenge 5 — Can we attribute the delay?

Inspect `driver_events.json`.

Look for:
- assignment,
- GPS pings,
- pickup,
- delivery.

> **Can we reliably tell when the driver arrived at the restaurant?**
**Answer: NO.** There is no explicit `arrived_at_restaurant` event logged. We can only infer arrival approximately from GPS pings near the restaurant coordinates.

### Required output

| Event / fact | Observed? | Source | Reliability concern |
|---|---|---|---|
| Driver assigned | Yes (Observed) | Dispatch API / `driver_events.json` | Reliable timestamp when dispatch engine assigned driver |
| Driver at restaurant | **No (Inferred)** | GPS pings in `driver_events.json` | **No explicit event logged**; must guess from GPS distance/geofence |
| Picked up | Yes (Observed) | `orders` table & `driver_events.json` | Relies on driver tapping 'picked up' in mobile app |
| Delivered | Yes (Observed) | `orders` table & `driver_events.json` | Relies on driver tapping 'delivered' at customer doorstep |


In [ ]:
# Challenge 5: Inspect driver events
first_driver = driver_events[0]
print('Driver:', first_driver['driver_id'])
display(pd.DataFrame(first_driver['events']).head(10))

# Flatten nested JSON
flat_events = []
for driver in driver_events:
    for event in driver['events']:
        flat_events.append({'driver_id': driver['driver_id'], **event})

driver_events_df = pd.DataFrame(flat_events)
print('\nEvent types breakdown:')
display(driver_events_df['type'].value_counts())

# Check for arrival event
has_arrival = 'arrived_at_restaurant' in driver_events_df['type'].values
print('Is there an explicit driver_arrived_at_restaurant event?', has_arrival)


# Final FDE synthesis — Executive Slide

### 1. Problem Size
- **Late Delivery Rate is 56.4%** among delivered orders with valid timestamps (843 out of 1,495 orders).
- The **median delay is 8.0 minutes** for late deliveries (mean: 10.5 minutes).
- 68 orders were cancelled, and 37 delivered orders lack completion timestamps.

### 2. Evidence
- The data shows that **kitchen prep time is strongly associated with delay (correlation: 0.83)**.
- Traffic (0.21) and distance (0.07) show weak correlation with delay.
- Customer complaints focus on moving ETAs, restaurant preparation delays, and cold food.

### 3. Trusted Sources
- **`flasheats.db` (`orders`)**: Authoritative for order created time, promised ETA, and delivery status.
- **Dispatch API**: Authoritative for driver dispatching and order fulfillment lifecycle.
- **`support_tickets.csv`**: Trusted for understanding customer friction and complaint causes.
- **`driver_events.json`**: Trusted for GPS movement and drop-off, but untrusted for restaurant arrival.

### 4. Uncertainty
- We cannot determine whether delays originate from slow kitchen cooking or long driver wait times outside restaurants.
- We cannot verify whether restaurant status updates in `restaurant_status.csv` reflect physical truth because updates often lag behind pickup.

### 5. Instrumentation Recommendation
- Implement an automatic **driver geofence arrival event** (50m from restaurant).
- Implement a digital **order ready button** on restaurant kitchen display tablets.
- Resolve data-logging failures that cause missing `actual_delivery_at` timestamps.

### 6. Decision: Would you build the AI delay predictor now?
**NO.** We would not build the AI delay predictor now.
- The root issue is operational latency in restaurant food preparation, not route estimation.
- An AI model trained without food-ready and store-arrival timestamps will perpetuate inaccurate ETAs and fail to reduce late deliveries.
- Fix kitchen operations and capture arrival timestamps before deploying predictive models.


In [ ]:
# Optional cleanup
try:
    con.close()
except:
    pass

try:
    api_proc.terminate()
except:
    pass

print("Session cleanup complete.")